In [1]:
!pip install pennylane


!pip uninstall -y torch torchvision
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 54.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 24.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 62.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 89.8 MB/s eta 0:00:00:00:0100:01
Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 79.0 MB/s eta 0:00:00

In [3]:
import os
import sys
import time
import random
import json
import gc
import logging
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import pennylane as qml
from pennylane import numpy as np

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from tqdm.auto import tqdm

from sklearn.manifold import TSNE
import seaborn as sns
import pandas as pd

class Config:
    MODE = 'FULL' 
    SEED = 42
    GRID_H = 2
    GRID_W = 2
    N_WIRES = GRID_H * GRID_W
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    DATA_ROOT = './data'
    OUTPUT_BASE = './outputs'
    RUN_DIR = None
    LOGGER = None

config = Config()

def seed_everything(random_seed=42):
    random.seed(random_seed)
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(random_seed)
        torch.cuda.manual_seed_all(random_seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def create_run_dir():
    current_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    run_folder_name = f"{current_timestamp}_Ablation_and_tSNE"
    config.RUN_DIR = os.path.join(config.OUTPUT_BASE, run_folder_name)
    if not os.path.exists(config.RUN_DIR):
        os.makedirs(config.RUN_DIR)

def setup_logger():
    log_file_path = os.path.join(config.RUN_DIR, "execution.log")
    logger = logging.getLogger("Experiment")
    logger.setLevel(logging.INFO)
    formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
    
    file_handler = logging.FileHandler(log_file_path)
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    
    stream_handler = logging.StreamHandler(sys.stdout)
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)
    
    config.LOGGER = logger

def save_to_json(data_dict, filename="final_results"):
    def json_serializer(obj):
        if isinstance(obj, (np.int64, np.int32)): return int(obj)
        if isinstance(obj, (np.float32, np.float64)): return float(obj)
        if isinstance(obj, torch.Tensor): return obj.item()
        return obj
    
    final_filename = f"{filename}_{config.MODE}.json"
    save_path = os.path.join(config.RUN_DIR, final_filename)
    with open(save_path, 'w') as f:
        json.dump(data_dict, f, default=json_serializer, indent=4)

def generate_grid_topology(height, width):
    h_pairs, v_pairs = [], []
    for row in range(height):
        for col in range(width - 1):
            h_pairs.append([row * width + col, row * width + col + 1])
    for row in range(height - 1):
        for col in range(width):
            v_pairs.append([row * width + col, (row + 1) * width + col])
    return h_pairs, v_pairs

def aux_download_datasets():
    datasets.MNIST(root=config.DATA_ROOT, train=False, download=True)
    datasets.FashionMNIST(root=config.DATA_ROOT, train=False, download=True)

def get_dataloader(dataset_name, data_size, batch_size, train=False, binary_mode=False):
    transform_pipeline = transforms.Compose([
        transforms.Resize((14, 14)),
        transforms.ToTensor()
    ])
    dataset_class = getattr(datasets, dataset_name)
    dataset = dataset_class(root=config.DATA_ROOT, train=train, download=True, transform=transform_pipeline)

    if binary_mode:
        mask = (dataset.targets == 0) | (dataset.targets == 1)
        dataset.data, dataset.targets = dataset.data[mask], dataset.targets[mask]

    if data_size != 'fullset':
        total_len = len(dataset)
        subset_size = min(data_size, total_len)
        rng = torch.Generator().manual_seed(config.SEED)
        indices = torch.randperm(total_len, generator=rng)[:subset_size]
        dataset = Subset(dataset, indices)

    kwargs = {'num_workers': 2, 'pin_memory': True} if torch.cuda.is_available() else {}
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=train, **kwargs)
    return loader

class LPQCNN(nn.Module):
    def __init__(self, n_filters, time_dyn, time_steps, n_classes=10, kernel_size=2, stride=1, noise_prob=0.0):
        super().__init__()
        self.n_filters = n_filters
        self.time_dyn = time_dyn
        self.time_steps = time_steps
        self.n_classes = n_classes 
        self.kernel_size = kernel_size
        self.stride = stride
        self.alpha = 0.5 
        self.noise_prob = noise_prob
        
        self.n_wires = config.N_WIRES
        self.h_pairs, self.v_pairs = generate_grid_topology(config.GRID_H, config.GRID_W)
        self.hamiltonian_params = nn.Parameter(torch.randn(n_filters, 6))
        
        if self.noise_prob > 0.0:
            self.dev = qml.device("default.mixed", wires=self.n_wires)
        else:
            self.dev = qml.device("default.qubit", wires=self.n_wires)
        
        @qml.qnode(self.dev, interface="torch", diff_method="backprop")
        def circuit(inputs, params):
            qml.AngleEmbedding(inputs * np.pi, wires=range(self.n_wires), rotation='Y')
            
            dt = self.time_dyn / self.time_steps
            jx_h, jy_h, jz_h, jx_v, jy_v, jz_v = params
            
            for _ in range(self.time_steps):
                for w in self.h_pairs:
                    qml.IsingXX(2*jx_h*dt, wires=w); qml.IsingYY(2*jy_h*dt, wires=w); qml.IsingZZ(2*jz_h*dt, wires=w)
                for w in self.v_pairs:
                    qml.IsingXX(2*jx_v*dt, wires=w); qml.IsingYY(2*jy_v*dt, wires=w); qml.IsingZZ(2*jz_v*dt, wires=w)
                
                if getattr(self, 'noise_prob', 0.0) > 0.0:
                    for w in range(self.n_wires):
                        qml.DepolarizingChannel(self.noise_prob, wires=w)
                        
            return [qml.expval(qml.PauliZ(i)) for i in range(self.n_wires)]
        
        self.qnode = circuit

        dim = 14
        out_dim = (dim - kernel_size) // stride + 1
        initial_input = n_filters * self.n_wires * out_dim * out_dim
        self.fc = nn.Linear(initial_input, n_classes)

    def forward(self, x):
        quantum_features = self.extract_quantum_features(x)
        return self.fc(quantum_features)

    def extract_quantum_features(self, x):
        batch_size, c, h, w = x.shape
        patches = F.unfold(x, kernel_size=self.kernel_size, stride=self.stride) 
        num_patches = patches.shape[-1]
        
        x_flat = patches.transpose(1, 2).reshape(-1, self.n_wires) * self.alpha 
        
        # Offload QNode inputs to CPU to bypass PennyLane's batched CUDA state mismatch bug
        # PyTorch Autograd natively tracks gradients across .cpu() and .to(device) calls.
        x_flat_cpu = x_flat.cpu()
        
        outputs = []
        for k in range(self.n_filters):
            p_cpu = self.hamiltonian_params[k].cpu()
            res = self.qnode(x_flat_cpu, p_cpu)
            
            if isinstance(res, (list, tuple)):
                res = torch.stack(res, dim=-1)
                
            outputs.append(res.to(config.DEVICE))
            
        out_tensor = torch.stack(outputs, dim=1) 
        out_tensor = out_tensor.view(batch_size, num_patches, self.n_filters, self.n_wires)
        
        return out_tensor.reshape(batch_size, -1).float()

def run_tsne_visualization(ds_name, config_dict, weight_path, num_samples=1000):
    config.LOGGER.info(f"\n--- Begin t-SNE extraction for {ds_name} ---")
    test_loader = get_dataloader(ds_name, "fullset", batch_size=128, train=False)
    
    model = LPQCNN(
        n_filters=config_dict['n_filters'], 
        time_dyn=config_dict['time'], 
        time_steps=config_dict['time_steps'], 
        n_classes=10
    ).to(config.DEVICE)
    
    try:
        loaded_data = torch.load(weight_path, map_location=config.DEVICE, weights_only=False)
        model.load_state_dict(loaded_data if not isinstance(loaded_data, nn.Module) else loaded_data.state_dict())
        config.LOGGER.info("Weights loaded successfully!")
    except Exception as e:
        config.LOGGER.error(f"Error loading model weights for t-SNE: {e}")
        return

    model.eval()
    features_list, labels_list = [], []
    
    config.LOGGER.info(f"Extracting {num_samples} features through LPQCNN...")
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(config.DEVICE)
            features = model.extract_quantum_features(images)
            
            features_list.append(features.cpu().numpy())
            labels_list.append(labels.numpy())
            
            if len(np.concatenate(features_list)) >= num_samples:
                break

    X_features = np.concatenate(features_list)[:num_samples]
    y_labels = np.concatenate(labels_list)[:num_samples]

    config.LOGGER.info("Running t-SNE dimensionality reduction...")
    tsne = TSNE(n_components=2, random_state=config.SEED, perplexity=30)
    tsne_results = tsne.fit_transform(X_features)
    
    df = pd.DataFrame({'TSNE_1': tsne_results[:, 0], 'TSNE_2': tsne_results[:, 1], 'Label': y_labels})
    
    classes_dict = {
        'FashionMNIST': ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'],
        'MNIST': [str(i) for i in range(10)]
    }
    class_names = classes_dict.get(ds_name, [str(i) for i in range(10)])
    df['Class'] = df['Label'].apply(lambda x: class_names[x])

    plt.figure(figsize=(12, 10))
    sns.scatterplot(
        x="TSNE_1", y="TSNE_2", hue="Class",
        palette=sns.color_palette("tab10", 10),
        data=df, legend="full", alpha=0.8, s=60
    )
    plt.legend(bbox_to_anchor=(1.02, 1), loc=2, borderaxespad=0.)
    plt.tight_layout()
    
    img_path = os.path.join(config.RUN_DIR, f"tSNE_{ds_name}.pdf")
    plt.savefig(img_path, dpi=300)
    plt.close()
    config.LOGGER.info(f"Visualization complete! Plot saved at: {img_path}")

if __name__ == "__main__":
    create_run_dir()
    setup_logger()
    seed_everything(config.SEED)
    
    config.LOGGER.info(f"[START] Executing t-SNE Pipeline | Computed on Device: {config.DEVICE}")
    aux_download_datasets()

    PRETRAINED_PATHS = {
        'FashionMNIST': '/kaggle/input/datasets/luongkhaichuong/lpqcnn-param/model_P1_FashionMNIST_FULL.pth', 
        'MNIST': '/kaggle/input/datasets/luongkhaichuong/lpqcnn-param/model_P1_MNIST_FULL.pth'
    }
    
    best_configs = {
        'MNIST': {'dataset': 'MNIST', 'time': 2.0, 'time_steps': 4, 'n_filters': 3}, 
        'FashionMNIST': {'dataset': 'FashionMNIST', 'time': 5.0, 'time_steps': 5, 'n_filters': 6}
    }
    
    for ds_name in ['FashionMNIST', 'MNIST']:
        if ds_name in best_configs and os.path.exists(PRETRAINED_PATHS[ds_name]):
            cfg = best_configs[ds_name]           
            
            run_tsne_visualization(ds_name, cfg, PRETRAINED_PATHS[ds_name], num_samples=1500)
            
            torch.cuda.empty_cache()
            gc.collect()
        else:
            config.LOGGER.warning(f"Skipping {ds_name} due to missing pretrained model weights at path!")

    config.LOGGER.info(f"\n[FINISH] Execution Pipeline Completed!")

2026-05-21 13:48:45 | INFO | [START] Executing t-SNE Pipeline | Computed on Device: cuda
2026-05-21 13:48:45 | INFO | [START] Executing t-SNE Pipeline | Computed on Device: cuda
2026-05-21 13:48:45 | INFO | 
--- Begin t-SNE extraction for FashionMNIST ---
2026-05-21 13:48:45 | INFO | 
--- Begin t-SNE extraction for FashionMNIST ---
2026-05-21 13:48:45 | INFO | Weights loaded successfully!
2026-05-21 13:48:45 | INFO | Weights loaded successfully!
2026-05-21 13:48:45 | INFO | Extracting 1500 features through LPQCNN...
2026-05-21 13:48:45 | INFO | Extracting 1500 features through LPQCNN...
2026-05-21 13:48:59 | INFO | Running t-SNE dimensionality reduction...
2026-05-21 13:48:59 | INFO | Running t-SNE dimensionality reduction...
2026-05-21 13:49:10 | INFO | Visualization complete! Plot saved at: ./outputs/2026-05-21_13-48-45_Ablation_and_tSNE/tSNE_FashionMNIST.pdf
2026-05-21 13:49:10 | INFO | Visualization complete! Plot saved at: ./outputs/2026-05-21_13-48-45_Ablation_and_tSNE/tSNE_Fashi